# Gett Failed Orders — Exploratory Analysis

## Assignment

1. Build up distribution of orders according to reasons for failure: cancellations before and after driver assignment, and reasons for order rejection. Analyse the resulting plot. Which category has the highest number of orders?
2. Plot the distribution of failed orders by hours. Is there a trend that certain hours have an abnormally high proportion of one category or another? What hours are the biggest fails? How can this be explained?
3. Plot the average time to cancellation with and without driver, by the hour. If there are any outliers in the data, it would be better to remove them. Can we draw any conclusions from this plot?
4. Plot the distribution of average ETA by hours. How can this plot be explained?
5. Hexagons. Using the h3 and folium packages, calculate how many sizes 8 hexes contain 80% of all orders from the original data sets and visualise the hexes, colouring them by the number of fails on the map.

## Load and Explore the Data

In [ ]:
import pandas as pd
import numpy as np

# Load both source files
orders_df = pd.read_csv("data_orders.csv")
offers_df = pd.read_csv("data_offers.csv")

In [ ]:
print("Orders shape:", orders_df.shape)
orders_df.sample(n=10, random_state=42)

In [ ]:
print("Offers shape:", offers_df.shape)
offers_df.sample(n=10, random_state=42)

In [ ]:
# Merge the two tables on the shared order key — similar to a SQL INNER JOIN
merged = orders_df.merge(right=offers_df, how="inner", on="order_gk")
merged.sample(n=10, random_state=42)

In [ ]:
# Replace binary flags with readable labels for easier interpretation
merged["driver_assigned"] = np.where(merged["is_driver_assigned_key"] == 1, "Yes", "No")
merged["fail_reason"] = np.where(merged["order_status_key"] == 4, "Client Cancelled", "System Reject")

# Drop the original encoded columns — we no longer need them
merged.drop(columns=["is_driver_assigned_key", "order_status_key"], inplace=True)

# Rename for clarity
merged.rename(columns={"order_datetime": "order_time"}, inplace=True)

merged.sample(n=10, random_state=42)

---
### Q1 — Failure Category Distribution
**Which failure category has the highest order count?**

In [ ]:
# Count orders grouped by driver assignment status and failure reason
q1_counts = merged.groupby(by=["driver_assigned", "fail_reason"])["order_gk"].count()
q1_counts

In [ ]:
# Pivot for a cleaner view, then plot
q1_pivot = merged.pivot_table(
    columns=["driver_assigned", "fail_reason"],
    values="order_gk",
    aggfunc="count"
)

q1_pivot.plot(
    kind="bar",
    figsize=(7, 7),
    legend=True,
    rot=0,
    title="Order Failures by Category"
)

q1_pivot

**Analysis:** The largest group is client cancellations *before* a driver was assigned (~13,435 orders). 
The second-largest is system rejections with no driver (~9,469), followed by client cancellations *after* driver assignment (~8,360). 
Only 4 orders were rejected by the system after a driver was already assigned — essentially negligible.

---
### Q2 — Failed Orders by Hour
**Is there a peak hour for failures? Does any category behave differently by hour?**

In [ ]:
# Extract hour from the time string
merged["order_hour"] = merged["order_time"].str.split(":").apply(lambda x: x[0])
merged[["order_time", "order_hour"]].sample(5, random_state=42)

In [ ]:
# Total failures per hour — overall view
merged.groupby("order_hour")["order_gk"].count().plot(
    figsize=(10, 7),
    xticks=range(0, 24),
    title="Total Failed Orders by Hour",
    legend=True
)

In [ ]:
# Break it down by category
q2_grouped = merged.groupby(["order_hour", "driver_assigned", "fail_reason"])["order_gk"].count()

q2_grouped.reset_index().pivot(
    index="order_hour",
    columns=["driver_assigned", "fail_reason"],
    values="order_gk"
).plot(
    figsize=(13, 7),
    xticks=range(0, 24),
    title="Failed Orders Per Hour — Broken Down by Category"
)

**Analysis:** Failures peak during evening/night hours (roughly 8 PM–midnight), which aligns with higher demand periods. 
Client cancellations without a driver assigned dominate across all hours. 
Notably, cancellations *with* a driver drop significantly during late-night hours — possibly because fewer drivers are active, so fewer driver-assigned rides are even initiated. 
The 4 system rejections with a driver all cluster around midnight.

---
### Q3 — Average Cancellation Time by Hour
**Does cancellation time differ by hour or by whether a driver was assigned?**

In [ ]:
# Average cancellation time (seconds) per hour, split by driver assignment
q3_grouped = merged.groupby(["order_hour", "driver_assigned"])["cancellations_time_in_seconds"].mean()

q3_grouped.reset_index().pivot(
    index="order_hour",
    columns="driver_assigned",
    values="cancellations_time_in_seconds"
).plot(
    figsize=(13, 7),
    xticks=range(0, 24),
    title="Avg. Time to Cancellation (seconds) by Hour — With vs. Without Driver"
)

**Analysis:** Cancellation time is consistently higher when a driver is already assigned — clients seem to wait longer before giving up once they know a driver is coming. 
Both lines peak around 3 AM, which could indicate slower driver response times or longer ETAs during off-peak hours. 
The pattern holds across every hour, suggesting this is a structural behavior rather than a time-of-day anomaly.

---
### Q4 — Average ETA by Hour
**How does estimated arrival time vary across the day?**

In [ ]:
# Average ETA per hour
merged.groupby("order_hour")["m_order_eta"].mean().plot(
    figsize=(14, 7),
    xticks=range(0, 24),
    title="Average ETA by Hour of Day"
)

**Analysis:** ETA is lowest during daytime hours (more drivers available) and rises in the evening — closely mirroring the failure rate curve from Q2. 
This suggests a direct link: longer ETAs lead to more client cancellations. 
When wait times spike, clients lose patience and cancel before pickup.

---
### Q5 — Hex Map: 80% Order Coverage
**How many H3 size-8 hexagons cover 80% of all orders? Visualize failures by hex.**

In [ ]:
import h3
import folium
import json
import geojson
import matplotlib
import matplotlib.colors
import h3.api.basic_int as h3_int

In [ ]:
# Assign each order to an H3 hexagon (resolution 8) based on pickup coordinates
merged["hex_id"] = merged.apply(
    lambda row: h3.latlng_to_cell(lat=row["origin_latitude"], lng=row["origin_longitude"], res=8),
    axis=1
)

In [ ]:
# Count orders per hexagon
hex_counts = merged.groupby("hex_id")["order_gk"].count().reset_index()
print("Total unique hexagons:", len(hex_counts))
hex_counts.sample(5, random_state=42)

In [ ]:
# Sort descending, compute cumulative percentage to find the 80% threshold
hex_counts.sort_values("order_gk", ascending=False, inplace=True)
total = hex_counts["order_gk"].sum()
hex_counts["cumulative_sum"] = hex_counts["order_gk"].cumsum()
hex_counts["cumulative_pct"] = 100 * hex_counts["cumulative_sum"] / total

top_80 = hex_counts[hex_counts["cumulative_pct"] <= 80]
print(f"Hexagons needed to cover 80% of all orders: {len(top_80)}")

In [ ]:
# Build GeoJSON features from hex IDs
def hex_to_geojson_feature(row):
    int_cell = int(row["hex_id"], 16)
    boundary = h3_int.cell_to_boundary(int_cell)
    # GeoJSON expects [lng, lat], h3 returns [lat, lng]
    coords = [[lng, lat] for lat, lng in boundary]
    return geojson.Feature(
        id=row["hex_id"],
        geometry={"type": "Polygon", "coordinates": [coords]},
        properties={"order_gk": row["order_gk"]}
    )

features = hex_counts.apply(hex_to_geojson_feature, axis=1).tolist()
geo_layer = json.dumps(geojson.FeatureCollection(features))

In [ ]:
# Build the folium map centered on the average pickup location
center_lat = merged["origin_latitude"].mean()
center_lng = merged["origin_longitude"].mean()

ride_map = folium.Map(location=[center_lat, center_lng], zoom_start=8.5, tiles="cartodbpositron")

# Color hexagons using a plasma colormap scaled to order count
cmap = matplotlib.colormaps["plasma"]
count_max = hex_counts["order_gk"].max()
count_min = hex_counts["order_gk"].min()

folium.GeoJson(
    data=geo_layer,
    style_function=lambda f: {
        "fillColor": matplotlib.colors.to_hex(
            cmap((f["properties"]["order_gk"] - count_min) / (count_max - count_min))
        ),
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.7
    }
).add_to(ride_map)

ride_map.save("map.html")

from IPython.display import HTML
HTML(ride_map._repr_html_())